# Build an Agent

**Prerequisites:** every notebook in this chapter, especially `02_conversations.ipynb`, `03_tokens_and_context.ipynb`, `06_function_calling.ipynb`, `08_asyncio.ipynb`

`resolve_tools()` (notebook 6) already has the shape of an agent loop: call the model, and if it wants a tool, run it and call again — repeating until there's a final answer instead of another tool request. What it's missing is memory: it builds a throwaway `messages` list from scratch every call, so nothing that loop does — a tool call, a validation failure, a corrected retry — ever becomes part of an actual conversation. This notebook fixes that: the same loop, living inside `Conversation`, so everything it does is real history you can inspect afterward.

**By the end you'll have:**
- Given `Conversation` an `asend_with_tools()` — the same call/execute/feed-back loop as `resolve_tools()`, async and with `schemas` for Pydantic validation, writing into the conversation's own `messages` instead of a throwaway list
- Proven that a validation failure and its corrected retry both land in `convo.messages`, not just the final answer
- Fixed `trim_to_budget()` (notebook 3) to handle tool-call messages correctly, and given it a way to preserve important facts instead of just discarding whatever it drops
- Built a small `Agent` wrapper — and given it a name, Loopy — so using this doesn't require hand-writing branching logic every time

## The agent loop, with memory

`resolve_tools()` (notebook 6) already is a minimal agent loop: call the model, execute whatever tool it asks for, feed the result back, call again — until the model returns a final answer instead of another tool request. The gap is memory: `resolve_tools()` builds its own `messages` list from scratch every call, so nothing that happens inside it — a tool call, a validation failure and its retry, a final answer — ever becomes part of a real `Conversation`.

Here's the shape of that loop — the part no diagram has shown yet, even though every piece of it (tool calls, retries, validation failures) has already been discussed landing in `messages`:

```
┌─────────────────────────────────────────────┐
│                                               │
▼                                               │
call the model                                 │
      │                                         │
      ├── API-level rejection (BadRequestError, notebook 6)
      │         │                               │
      │         ▼                               │
      │   append a generic "try again"          │
      │   user message                          │
      │         │                               │
      │         └───────────────────────────────┘
      │            (loops back up to "call the model")
      ▼ (response received, no rejection)
did it request a tool?
      │
     no ──▶ append final answer, return   ← the only way out of the loop
      │
     yes
      │
      ▼
   ┌───────────────────────┐
   │                        │
   ▼                        │
arguments good?             │
      │                     │
     no ──▶ append the      │
      │      validation     │
      │      error as the   │
      │      tool's result  │
      │            │        │
      │            └────────┘
      │        (loops back up to "arguments good?")
     yes
      │
      ▼
run the function, append its real result
      │
      ▼
back to "call the model" (at the very top)
```

Two different failure branches, both looping back rather than giving up: an API-level rejection (no `tool_call.id` yet, so the retry is a plain `user` message) loops back to the very first "call the model," and a validation failure (a real `tool_call.id` exists, so the error goes back as that tool's own result) loops back to "arguments good?" — the exact two categories notebook 6 built `resolve_tools()` around. Neither is a separate loop with its own budget — both are the *same* outer loop, just re-entered at a different point, drawing down the same shared `retries` count as everything else until the answer is "yes" (or "no rejection") or the budget runs out. It's a loop with an exit condition, not a fixed number of steps — it keeps going around for as long as there's a tool call to make or a failure to recover from, and only stops once a turn comes back with no tool call at all (or the retry budget runs out). `resolve_tools()` already runs exactly this loop; `asend_with_tools()` below is the same loop, just writing every step — successful or not — into a real `Conversation` instead of a throwaway list.

Two tools an assistant would plausibly offer for real:

`asend_with_tools()` is exactly the loop diagrammed above, bound to `Conversation`: call, and if the model wants tools, run them and feed the results back — using the same retry design as `resolve_tools()` from notebook 6, including the optional `schemas` argument for Pydantic validation. If the model requests more than one tool in a single turn (it can, entirely on its own — notebook 6 covers this), they run concurrently via `asyncio.to_thread` + `gather`, exactly the pattern notebook 8 already proved is faster; that's not re-measured here. The one thing that's new is *where* every step gets written:

```python
async def asend_with_tools(self, content, tools, functions, schemas=None, retries=3, model=DEFAULT_MODEL):
    self.messages.append({"role": "user", "content": content})
    for _ in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=model, messages=self.messages, tools=tools
            )
        except BadRequestError:
            self.messages.append({
                "role": "user",
                "content": "That tool call could not be processed. Try again, calling one of the available tools correctly.",
            })
            continue

        message = response.choices[0].message
        self.messages.append(message)

        if not message.tool_calls:
            return message.content

        async def run(tool_call):
            try:
                return await asyncio.to_thread(execute_tool_call, tool_call, functions, schemas)
            except Exception as e:
                return f"Error: {e}"

        results = await asyncio.gather(*(run(tc) for tc in message.tool_calls))
        for tool_call, result in zip(message.tool_calls, results):
            self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    raise ValueError(f"Failed to get a final answer after {retries} attempts")
```

Every `self.messages.append(...)` call is the difference from `resolve_tools()` — the user turn, every tool-call attempt (successful or not), every tool result, the corrective nudge on an API-level rejection, the final answer: all of it lands in the same list `Conversation` already uses for plain text. (Notebook 6 covers *why* the corrective message stays generic rather than extracting the safe part of Groq's error — that discussion belongs there, next to where `resolve_tools()` and the rest of the API-level-failure handling was actually built.)

Nothing in the prompt named either tool, yet the model requested both — sometimes in the same turn, sometimes across two separate tool round trips, depending on how it chooses to work through the request. Either way, `convo.messages` grew well past the 2 messages a plain exchange would leave: the user turn, one or two rounds of tool-call-plus-result pairs, and the final answer — all genuinely in history, not thrown away after the answer was printed.

## Proving failure recovery is real history too

A validation failure recovering (notebook 6) is real, but reproducing one live and on demand isn't reliable — the model rarely gets a schema-constrained argument wrong when the schema is this simple. So, same as notebook 6's divide-by-zero demo, simulate the mistake by hand: seed a fresh `Conversation` as if the model had already tried `convert_temperature` with `"Fahrenheit"` instead of `"F"`, and run that fabricated call through the real `execute_tool_call()` + `SCHEMAS` to see the real validation error:

That error is already sitting in `convo2.messages` as a real `role="tool"` result, tied to `call_1` — not printed and discarded. Now let the model see its own mistake and retry, for real. `asend_with_tools()` can't resume this exact exchange (it always starts by appending a *new* user turn), so this walks through its retry loop by hand, directly against `convo2.messages` — the same two categories from notebook 6, still landing in the same list:

The corrected retry usually supplies a valid unit (`"F"`) on the very next attempt and finishes in two rounds: a real tool result, then a natural-language final answer. Occasionally the corrective error text itself (which includes a Pydantic documentation URL) confuses the model into generating broken tool-call syntax, triggering the *other* failure category — an API-level rejection — before a clean retry follows. Either way, every attempt, error, and correction is a real entry being appended to `convo2.messages`, not a side effect that vanishes once the final answer arrives.

The bad call, the validation error, the corrected retry, the tool result, the final answer — the entire recovery is sitting right there in `convo2.messages`, inspectable after the fact. That's the actual point of moving this loop into `Conversation`: `resolve_tools()` proved the mechanism works in notebook 6, but its `messages` list was thrown away the moment the function returned. Here, nothing is.

## A wrapper, not a script

Every example so far has meant hand-writing `if tools: await asend_with_tools(...) else: send(...)`, plus printing labels around it, every single time. That's not an agent, that's a debugging harness. `Agent` in `chat.py` owns that branching and the formatting once, so using it doesn't require re-deriving either:

```python
class Agent:
    def __init__(self, name="Agent", system=None, tools=None, functions=None, schemas=None):
        self.name = name
        self.conversation = Conversation(system=system)
        self.tools = tools
        self.functions = functions
        self.schemas = schemas

    async def ask(self, message):
        if self.tools:
            answer = await self.conversation.asend_with_tools(message, self.tools, self.functions, self.schemas)
        else:
            answer = await asyncio.to_thread(self.conversation.send, message)
        print(f"{self.name}: {answer}")
```

One method. The caller never touches `asend_with_tools` vs. `send`, never checks whether a reply came from a tool round trip or plain text — `Agent` decided that once, internally. `name` is just what gets printed — pass in whatever this particular agent should be called. There's no echo of the question itself: `message` is already headed into `self.conversation.messages` via `asend_with_tools`/`send`, so printing it back would just be repeating what the person calling `ask()` already knows they typed. `ask()` doesn't return anything either — its job is to print and update history, not hand back a value for a notebook cell to auto-display. `schemas` is optional and defaults to `None`, same as `resolve_tools()` in notebook 6 — pass one in and bad tool arguments get caught by Pydantic before the tool function ever runs; omit it and you get the old, unchecked behavior. The plain-text path runs through `asyncio.to_thread` rather than a native async call — `send()` is a synchronous, blocking function, and that's the only way to run one without blocking the event loop while `ask()` awaits it.

Eight real turns this time — four ordinary factual/tool turns interleaved with four turns that actually state something worth remembering: a preference, a constraint, a decision, and an open question. That's the exact set of categories the `remember=True` prompt above is built to recognize, so run them for real and see what `notes` ends up holding. Nothing in any of these prompts names a tool:

A constraint, stated plainly, with no tool involved:

A decision, again with no tool needed:

And an open question, still unresolved:

One last turn, asking Loopy to recall the factual stuff from earlier:

Eight calls, no branching visible at any call site — four ordinary tool/factual turns and four turns that actually stated a preference, a constraint, a decision, and an open question, and the final answer still correctly recalls the favorite number and both cities' times. A purely text-only conversation of 8 turns would sit at 17 messages (1 system + 8 user + 8 assistant); `loopy.conversation.messages` landed at 26 — the extra 9 are every tool-call request and every tool result from the three tool-using turns (Tokyo, London, C-to-F), still sitting right there in history.

## Trimming an agent's history without losing what matters

`trim_to_budget()` (notebook 3) was built for plain text — every message a dict, `user`/`assistant` strictly alternating. Loopy's history breaks both assumptions: a tool-call message carries no text at all — the content is gone and the `tool_calls` are the payload — and a single "turn" can be several messages long (a tool-call request, its result, then the final answer). `trim_to_budget()` has been designed keeping this in mind instead of assuming exactly two messages, so a tool round trip gets dropped as a whole — never leaving an orphaned tool result behind. Applied directly to `loopy.conversation.messages`, at increasing budgets:

No crash, and every kept count corresponds to whole turns — 1 (just the system message), 3, 3, 9 — never a count that would split a tool call from its result. But dropping turns outright still means losing whatever they contained permanently. `Conversation.trim()` now takes a `remember=True` flag: before dropping anything, it runs the turns about to be lost through `render_transcript()` and `extract()`, and keeps whatever comes back in a new `self.notes` list — a separate attribute, not mixed into `messages`, so nothing here changes what the model actually sees on the next call:

```python
def trim(self, max_tokens, reserve_turns=2, remember=False):
    trimmed = trim_to_budget(self.messages, max_tokens, reserve_turns)

    if remember:
        dropped = [m for m in self.messages if not any(m is kept for kept in trimmed)]
        transcript = render_transcript(dropped)
        if transcript:
            memory = extract(
                "Extract only genuine decisions, constraints, preferences, or open/unanswered "
                "questions worth remembering long after this excerpt is gone. A stated personal "
                "preference counts even if it seems minor (a favorite number, color, or similar) "
                "— what matters is that the user stated something about themselves, not how "
                "consequential it is. Do NOT include factual answers, tool results, or a recap "
                "of what was discussed — a turn that's just a factual query with no decision, "
                "constraint, preference, or open question behind it should produce nothing at "
                "all. Return JSON with a single field \"key_facts\": a list of short strings, "
                f"one per fact. If nothing qualifies, use an empty list.\n\n{transcript}",
                ConversationMemory,
            )
            self.notes.extend(memory.key_facts)

    self.messages = trimmed
```

`dropped` is computed by identity (`m is kept`), not equality — the same message objects that survived trimming are excluded, everything else goes to `render_transcript()`. The prompt is deliberately strict: it isn't enough for something to be *mentioned*, it has to be a decision, constraint, preference, or an open question — a plain factual query like "what time is it in Tokyo" should produce nothing on its own, even though it's real content that's about to be lost.

Seven of the eight turns are gone from `messages` — only the final recall survives. But `loopy.conversation.notes` ends up holding exactly four things: the favorite number (a preference), being vegetarian (a constraint), booking London over Tokyo (a decision), and being undecided about currency conversion (an open question) — one fact per category, extracted correctly out of a much larger excerpt. The Tokyo time, the London time, and the F-to-C conversion are just as gone, and none of them show up in `notes` at all, because none of them were a decision, constraint, preference, or open question — just facts nobody asked to keep. `self.notes` is deliberately inert here — nothing re-injects it into a future `send()`/`asend_with_tools()` call. Deciding how (and whether) a note like this should resurface is a real policy question, and policy is exactly what `04_memory` is for; this just proves the right facts survive the trim, and only the right facts.

## What you built

✓ Gave `Conversation` an `asend_with_tools()` — the same call/execute/feed-back loop as `resolve_tools()` (notebook 6), now writing into the conversation's own `messages` instead of a throwaway list

✓ Proved a normal multi-tool round trip really grows `convo.messages` well past a plain 2-message exchange, not just returns an answer

✓ Proved a validation failure and its corrected retry are both real entries in `convo2.messages` — bad call, error, correction, and final answer all inspectable afterward, not just the final answer

✓ Fixed `trim_to_budget()` to group by turn instead of assuming exactly two messages, so it no longer crashes on tool-call messages or splits a tool result from its call

✓ Gave `Conversation.trim()` a `remember=True` option that extracts key facts from turns about to be dropped and keeps them in `self.notes`, separate from `messages` — proved it correctly captures a preference, a constraint, a decision, and an open question out of a longer excerpt, while producing nothing at all for the plain factual turns mixed in alongside them

✓ Built `Agent` in `chat.py`, named an instance Loopy, so calling code never branches on tools vs. plain text itself, with the plain-text path sharing `resolve_tools()`'s retry design (notebook 6)

This closes `01_foundations`. Every primitive here — `chat()`, `Conversation`, `trim_to_budget()`, `PromptTemplate`, `extract()`, `execute_tool_call()`/`resolve_tools()`, `stream_chat()`, and now `Agent` — is what `02_agent_runtime` composes into a fuller agent: something that plans across many steps and decides what to do next, instead of a fixed call/execute/feed-back loop with a bounded retry budget.